# Patch Concurrency with CUDA Streams

Out next step is improving performance of our patch-based algorithm.
The main reason of the performance loss observed is **serialization** on the kernel level.
To fix this we employ CUDA streams.

## CUDA Stream Recap

* CUDA supports different streams which can be seen as simple work queues.
* Operations enqueued or put into the streams are memory operations (`cudaMemcpy`, `cudaMemcpyAsync`, ..., `cudaMemPrefetchAsync`, ...), kernels, and others.
* Stream 0 is called the **default stream**.
* Operations inside one stream are executed
  * **in order** of their submission, and
  * **without overlap** with other operations of the **same stream**.
* Operations between different streams **may** overlap.
* The default stream is **blocking** with respect to other streams.

Let's look at an example.

```cpp
cudaMemcpyAsync(..., cudaMemcpyDeviceToHost);
kernelA<<<grid, block>>>();
kernelB<<<grid, block>>>();
kernelC<<<grid, block>>>();
cudaMemcpyAsync(..., cudaMemcpyHostToDevice);
```


Without specification, the implicitly targeted stream is the default stream, or stream 0.
Explicitly targeting this stream is also possible and yields equivalent behavior.

```cpp
cudaMemcpyAsync(..., cudaMemcpyDeviceToHost, 0);
kernelA<<<grid, block, 0, 0>>>();
kernelB<<<grid, block, 0, 0>>>();
kernelC<<<grid, block, 0, 0>>>();
cudaMemcpyAsync(..., cudaMemcpyHostToDevice, 0);
```


The figure below shows a possible execution, similar to what Nsight Systems might show in its timeline visualization.
Note the ordering and the absence of overlap.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-default.png" width="960" style="background-color:white" alt="Default Stream Visualization">
    </td>
  </tr>
</table>

Let's enqueue kernels B and C into separate streams.

```cpp
cudaMemcpyAsync(..., cudaMemcpyDeviceToHost);
kernelA<<<grid, block>>>();
kernelB<<<grid, block, 0, stream_1>>>();
kernelC<<<grid, block, 0, stream_2>>>();
cudaMemcpyAsync(..., cudaMemcpyHostToDevice);
```


The figure below shows a possible execution.
Note the ordering within streams, the blocking nature of the default stream and the overlap between nondefault streams.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-nondefault.png" width="960" style="background-color:white" alt="Nondefault Stream Visualization">
    </td>
  </tr>
</table>

The last example displays adding all kernels into separate streams.

```cpp
cudaMemcpyAsync(..., cudaMemcpyDeviceToHost);
kernelA<<<grid, block, 0, stream_1>>>();
kernelB<<<grid, block, 0, stream_2>>>();
kernelC<<<grid, block, 0, stream_3>>>();
cudaMemcpyAsync(..., cudaMemcpyHostToDevice);
```


The figure below shows a possible execution.
Note the arbitrary execution order between streams.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-nondefault-2.png" width="960" style="background-color:white" alt="Nondefault Stream Visualization">
    </td>
  </tr>
</table>

## Streams for Patches

All patches can be processed independently and in parallel since there are **no (read-write) data dependencies** between them.
To enable parallel execution, we assign **one CUDA stream per patch** and launch each patch’s kernel on its own stream.

### 1. Create and Destroy Streams

We create one stream per patch.
Having a separate array managing the streams of this application has certain advantages, but we keep it simple for now and add the stream handle to the `Patch` struct.

```cpp
struct Patch {
    // ... as before

    // patch stream
    cudaStream_t stream;
};
```


In out setup phase, we can create the patch streams.

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    auto &patch = patches[patchIdx];

    // ... as before

    // create stream
    checkCudaError(cudaStreamCreate(&patch.stream));       
}
```


At the end of our application, we destroy the streams and free the array holding them.

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    auto &patch = patches[patchIdx];
    checkCudaError(cudaStreamDestroy(patch.stream));
}
```


Note that the creation function takes a **pointer** to a stream while the destroy function takes a **reference**.

### 2. Use Streams to Launch Kernels

After preparing the streams, the final step is using them to express that our kernels may run in parallel.
We do this by providing the corresponding stream as the fourth argument in the launch configuration.

```cpp
auto work = [&](size_t it) {
    for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
        const auto &patch = patches[patchIdx];

        stencil2D<<<patch.gridSize, patch.blockSize, 0, patch.stream>>>(...);
    }

    checkCudaError(cudaDeviceSynchronize());

    std::swap(u, uNew);

    // ...
};
```


Note the additional synchronization before swapping.
We use `cudaDeviceSynchronize` which synchronizes with **all streams** (of the current GPU).
Alternatively, multiple calls to `cudaStreamSynchronize` can be used.

In any case, additional synchronization is necessary here to ensure that all streams are in the same time step to prevent race conditions since
* all kernels work on the same global array amd
* streams may progress at varying speeds.

### 3. Optional: Prefetch in Streams

Let's look at one potential optimization.
We start by remembering the behavior we started out with.
Assuming 24 rows (22 inner rows), the behavior might be similar to the one illustrated below.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-prefetch-base.png" width="960" style="background-color:white" alt="Prefetch in Stream Visualization">
    </td>
  </tr>
</table>

Then, we parallelized the kernels by distributing them onto streams which might yield the behavior shown in the next figure.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-prefetch-comp.png" width="960" style="background-color:white" alt="Prefetch in Stream Visualization">
    </td>
  </tr>
</table>

The optimization we want to apply for the **first time step** is splitting up the prefetch operation too, which might produce the final output shown below.

<table>
  <tr>
    <td align="center">
      <img src="./img/streams-prefetch-full.png" width="960" style="background-color:white" alt="Prefetch in Stream Visualization">
    </td>
  </tr>
</table>

To implement the optimization, we can follow this logic:

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    const auto &patch = patches[patchIdx];

    auto startIdx = (patch.globalInnerBeginY - 1) * globalNumCellsX;
    auto endIdx = (patch.globalInnerEndY + 1) * globalNumCellsX;
    auto size = (endIdx - startIdx) * sizeof(double);

    checkCudaError(cudaMemPrefetchAsync(u + startIdx, size, deviceId, stream));
    checkCudaError(cudaMemPrefetchAsync(uNew + startIdx, size, deviceId, stream));
}
```

Note that to see an effect on timings, we would need to rework how our synchronization works.
Currently, our way of using host-based timers and the required synchronization effectively prevents overlapping.

## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/05-streams/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/05-streams/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/05-streams/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/05-streams/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/05-streams/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/05-streams/stencil-2d-medium.cu ../src/05-streams/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/05-streams/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/05-streams/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/05-streams/stencil-2d-easier.cu ../src/05-streams/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/05-streams/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/05-streams/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/05-streams/stencil-2d-solution.cu ../src/05-streams/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [05-streams/stencil-2d.cu](../src/05-streams/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/05-streams ../src/05-streams/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/05-streams 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/05-streams $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on a **single A100** GPU.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:1 \
    --time 00:05:00 --wait \
    --output=../output/05-streams.out --error=../output/05-streams.err \
    --wrap="../build/05-streams $((32 * 1024)) 256 2 8 0"

cat ../output/05-streams.out

The next cell performs profiling with Nsight Systems by submitting a batch job.

The profile is then available at [profiles/05-streams.nsys-rep](../profiles/05-streams.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/05-streams-nsys.out --error=../output/05-streams-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/05-streams \
        ../build/05-streams $((32 * 1024)) 256 2 8 0"

cat ../output/05-streams-nsys.out

## Next Step

The next step is executing the per-patch kernels on multiple GPUs in [06](./06-mgpu.ipynb).